<a href="https://colab.research.google.com/github/meredith224/Pandas_DE_Academy/blob/main/revops_related_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Set display options so nothing gets truncated
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# ============================================================
# SECTION 1: BUILD FAKE DATASETS
# ============================================================
# In interviews they'll hand you CSVs — here we simulate them.
# We're creating 4 tables that mirror what you'd see in a
# CRM + billing system (Salesforce, HubSpot, Stripe, etc.)
# ============================================================

np.random.seed(42)
n_customers = 500

# --- CUSTOMERS TABLE ---
# Think: HubSpot contacts or Salesforce accounts
channels = ['Google Ads', 'Facebook Ads', 'Organic Search', 'Referral', 'LinkedIn Ads']
segments = ['SMB', 'Mid-Market', 'Enterprise']

customers = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'signup_date': pd.date_range('2024-01-01', periods=n_customers, freq='4h'),
    'acquisition_channel': np.random.choice(channels, n_customers, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'segment': np.random.choice(segments, n_customers, p=[0.60, 0.30, 0.10]),
    'is_churned': np.random.choice([0, 1], n_customers, p=[0.72, 0.28])
})

customers['signup_month'] = customers['signup_date'].dt.to_period('M')

print(f"Customers table: {customers.shape}")
customers.head()

Customers table: (500, 6)


,customer_id,signup_date,acquisition_channel,segment,is_churned,signup_month
0,1,2024-01-01 00:00:00,Facebook Ads,Mid-Market,0,2024-01
1,2,2024-01-01 04:00:00,LinkedIn Ads,SMB,0,2024-01
2,3,2024-01-01 08:00:00,Organic Search,SMB,1,2024-01
3,4,2024-01-01 12:00:00,Organic Search,Mid-Market,1,2024-01
4,5,2024-01-01 16:00:00,Google Ads,Mid-Market,1,2024-01


In [ ]:
# --- ORDERS TABLE ---
# Think: Stripe charges or Salesforce opportunities (Closed Won)
# Each customer has 1-15 orders depending on segment

orders_list = []
order_id = 1

for _, row in customers.iterrows():
    # Enterprise buys more, SMB buys less — realistic skew
    if row['segment'] == 'Enterprise':
        n_orders = np.random.randint(3, 15)
        avg_price = np.random.uniform(800, 3000)
    elif row['segment'] == 'Mid-Market':
        n_orders = np.random.randint(2, 10)
        avg_price = np.random.uniform(200, 800)
    else:  # SMB
        n_orders = np.random.randint(1, 6)
        avg_price = np.random.uniform(30, 250)

    for i in range(n_orders):
        orders_list.append({
            'order_id': order_id,
            'customer_id': row['customer_id'],
            'order_date': row['signup_date'] + pd.Timedelta(days=np.random.randint(0, 180)),
            'revenue': round(np.random.normal(avg_price, avg_price * 0.2), 2)
        })
        order_id += 1

orders = pd.DataFrame(orders_list)
orders['revenue'] = orders['revenue'].clip(lower=5)  # no negative or near-zero orders
orders['order_month'] = orders['order_date'].dt.to_period('M')

print(f"Orders table: {orders.shape}")
orders.head()

Orders table: (2179, 5)


,order_id,customer_id,order_date,revenue,order_month
0,1,1,2024-06-04,511.45,2024-06
1,2,1,2024-02-06,475.21,2024-02
2,3,1,2024-02-13,365.99,2024-02
3,4,1,2024-03-27,632.78,2024-03
4,5,1,2024-03-10,829.41,2024-03


In [ ]:
# --- AD SPEND TABLE ---
# Think: data exported from Google Ads / Facebook Ads Manager
# Monthly spend per paid channel

months = pd.period_range('2024-01', '2024-07', freq='M')
paid_channels = ['Google Ads', 'Facebook Ads', 'LinkedIn Ads']

ad_spend_list = []
for month in months:
    for channel in paid_channels:
        base = {'Google Ads': 12000, 'Facebook Ads': 8000, 'LinkedIn Ads': 5000}[channel]
        ad_spend_list.append({
            'month': month,
            'channel': channel,
            'spend': round(base + np.random.normal(0, base * 0.15), 2)
        })

ad_spend = pd.DataFrame(ad_spend_list)

print(f"Ad Spend table: {ad_spend.shape}")
ad_spend.head(9)

Ad Spend table: (21, 3)


,month,channel,spend
0,2024-01,Google Ads,13303.45
1,2024-01,Facebook Ads,9548.91
2,2024-01,LinkedIn Ads,6940.93
3,2024-02,Google Ads,9840.49
4,2024-02,Facebook Ads,8838.70
5,2024-02,LinkedIn Ads,4692.65
6,2024-03,Google Ads,13410.05
7,2024-03,Facebook Ads,6235.41
8,2024-03,LinkedIn Ads,4052.92


In [ ]:
# --- PIPELINE / DEALS TABLE ---
# Think: Salesforce Opportunities at various stages
# This is for funnel conversion + pipeline velocity questions

stages = ['Lead', 'MQL', 'SQL', 'Opportunity', 'Proposal', 'Closed Won', 'Closed Lost']
n_deals = 800

pipeline = pd.DataFrame({
    'deal_id': range(1, n_deals + 1),
    'owner': np.random.choice(['Alice', 'Bob', 'Carol', 'Dan'], n_deals),
    'stage': np.random.choice(stages, n_deals, p=[0.20, 0.18, 0.15, 0.14, 0.10, 0.13, 0.10]),
    'deal_value': np.round(np.random.lognormal(mean=7, sigma=1.2, size=n_deals), 2),
    'created_date': pd.date_range('2024-01-01', periods=n_deals, freq='3h'),
    'days_in_stage': np.random.randint(1, 90, n_deals)
})

print(f"Pipeline table: {pipeline.shape}")
pipeline.head()

Pipeline table: (800, 6)


,deal_id,owner,stage,deal_value,created_date,days_in_stage
0,1,Bob,Lead,198.10,2024-01-01 00:00:00,38
1,2,Dan,MQL,1759.26,2024-01-01 03:00:00,43
2,3,Carol,Closed Lost,2966.65,2024-01-01 06:00:00,87
3,4,Carol,Opportunity,513.89,2024-01-01 09:00:00,14
4,5,Dan,MQL,1555.33,2024-01-01 12:00:00,6


In [ ]:
# ============================================================
# SECTION 2: AOV (Average Order Value)
# ============================================================
# INTERVIEW Q: "What's our AOV? How does it differ by segment?"
# This is usually the warm-up question.
# ============================================================

# Overall AOV — dead simple
overall_aov = orders['revenue'].mean()
print(f"Overall AOV: ${overall_aov:,.2f}")
print(f"Median Order Value: ${orders['revenue'].median():,.2f}")
print()  # median matters — AOV gets skewed by whales

# AOV by segment — need to merge orders with customers first
# This merge pattern comes up in EVERY interview
orders_enriched = orders.merge(customers[['customer_id', 'segment', 'acquisition_channel']], on='customer_id')

aov_by_segment = (
    orders_enriched
    .groupby('segment')['revenue']
    .agg(['mean', 'median', 'count'])
    .rename(columns={'mean': 'aov', 'median': 'median_ov', 'count': 'num_orders'})
    .sort_values('aov', ascending=False)
)

print("AOV by Segment:")
aov_by_segment

Overall AOV: $629.83
Median Order Value: $291.39

AOV by Segment:


,aov,median_ov,num_orders
segment,,,
Enterprise,2046.44,1991.95,402
Mid-Market,532.96,530.04,785
SMB,132.41,126.12,992


In [ ]:
# ============================================================
# SECTION 4: CAC (Customer Acquisition Cost)
# ============================================================
# INTERVIEW Q: "Calculate CAC per channel. Which is most efficient?"
# CAC = Total Spend / Number of Customers Acquired
# ============================================================

# Step 1: total spend per channel (across all months)
total_spend_by_channel = ad_spend.groupby('channel')['spend'].sum()
print("Total Ad Spend by Channel:")
print(total_spend_by_channel)
print()

# Step 2: count customers acquired per channel
customers_by_channel = customers['acquisition_channel'].value_counts()

# Step 3: calculate CAC — only for PAID channels
# Common interview trap: don't include organic/referral in CAC calc
cac = (total_spend_by_channel / customers_by_channel).dropna().sort_values()

print("CAC by Paid Channel:")
for channel, cost in cac.items():
    print(f"  {channel}: ${cost:,.2f}")

Total Ad Spend by Channel:
channel
Facebook Ads   57722.85
Google Ads     89711.62
LinkedIn Ads   35425.51
Name: spend, dtype: float64

CAC by Paid Channel:
  Facebook Ads: $489.18
  Google Ads: $582.54
  LinkedIn Ads: $644.10


In [ ]:
# ============================================================
# SECTION 5: LTV (Customer Lifetime Value)
# ============================================================
# INTERVIEW Q: "Calculate LTV. What's our LTV:CAC ratio?"
# Simple LTV = avg revenue per customer over their lifetime
# ============================================================

# LTV per customer = sum of all their orders
ltv_per_customer = orders.groupby('customer_id')['revenue'].sum().reset_index()
ltv_per_customer.columns = ['customer_id', 'ltv']

# Merge back to get segment + channel info
ltv_enriched = ltv_per_customer.merge(customers[['customer_id', 'segment', 'acquisition_channel']], on='customer_id')

print("Overall LTV Stats:")
print(f"  Mean LTV:   ${ltv_enriched['ltv'].mean():,.2f}")
print(f"  Median LTV: ${ltv_enriched['ltv'].median():,.2f}")
print()

# LTV by segment — this is the key insight interviewers want
ltv_by_segment = (
    ltv_enriched
    .groupby('segment')['ltv']
    .agg(['mean', 'median', 'std', 'count'])
    .sort_values('mean', ascending=False)
)

print("LTV by Segment:")
ltv_by_segment

Overall LTV Stats:
  Mean LTV:   $2,744.78
  Median LTV: $629.40

LTV by Segment:


,mean,median,std,count
segment,,,,
Enterprise,18281.49,15259.82,10993.94,45
Mid-Market,3099.09,2763.95,1786.80,135
SMB,410.46,368.62,265.01,320


In [ ]:
# ============================================================
# SECTION 6: LTV:CAC RATIO
# ============================================================
# INTERVIEW Q: "Is our acquisition profitable? Where should
#               we invest more?"
# Rule of thumb: LTV:CAC > 3 is healthy, < 1 means you're
# losing money on every customer you acquire.
# ============================================================

# Average LTV per paid channel
ltv_by_channel = (
    ltv_enriched[ltv_enriched['acquisition_channel'].isin(paid_channels)]
    .groupby('acquisition_channel')['ltv']
    .mean()
)

# Build the LTV:CAC comparison
ltv_cac = pd.DataFrame({
    'avg_ltv': ltv_by_channel,
    'cac': cac
}).dropna()

ltv_cac['ltv_cac_ratio'] = ltv_cac['avg_ltv'] / ltv_cac['cac']
ltv_cac['profitable'] = ltv_cac['ltv_cac_ratio'] > 3  # industry standard benchmark

print("LTV:CAC Ratio by Channel:")
ltv_cac.sort_values('ltv_cac_ratio', ascending=False)

LTV:CAC Ratio by Channel:


,avg_ltv,cac,ltv_cac_ratio,profitable
Facebook Ads,2566.65,489.18,5.25,True
LinkedIn Ads,3000.55,644.10,4.66,True
Google Ads,2687.22,582.54,4.61,True


In [ ]:
# ============================================================
# SECTION 8: CHURN RATE
# ============================================================
# INTERVIEW Q: "What's our churn rate? Which segment churns most?"
# Churn Rate = # churned customers / total customers
# ============================================================

# Overall churn
overall_churn = customers['is_churned'].mean()
print(f"Overall Churn Rate: {overall_churn:.1%}")
print()

# Churn by segment
churn_by_segment = (
    customers
    .groupby('segment')
    .agg(
        total_customers=('customer_id', 'count'),
        churned=('is_churned', 'sum'),
        churn_rate=('is_churned', 'mean')
    )
    .sort_values('churn_rate', ascending=False)
)

print("Churn by Segment:")
print(churn_by_segment)
print()

# Churn by channel — helps answer "is a channel bringing us bad leads?"
churn_by_channel = (
    customers
    .groupby('acquisition_channel')
    .agg(
        total=('customer_id', 'count'),
        churned=('is_churned', 'sum'),
        churn_rate=('is_churned', 'mean')
    )
    .sort_values('churn_rate', ascending=False)
)

print("Churn by Acquisition Channel:")
churn_by_channel

Overall Churn Rate: 30.0%

Churn by Segment:
            total_customers  churned  churn_rate
segment                                         
SMB                     320      103        0.32
Enterprise               45       12        0.27
Mid-Market              135       35        0.26

Churn by Acquisition Channel:


,total,churned,churn_rate
acquisition_channel,,,
LinkedIn Ads,55,22,0.40
Google Ads,154,50,0.32
Referral,76,24,0.32
Facebook Ads,118,35,0.30
Organic Search,97,19,0.20


In [ ]:
# ============================================================
# SECTION 9: MONTHLY RECURRING REVENUE (MRR)
# ============================================================
# INTERVIEW Q: "Show me MRR trend. Are we growing?"
# MRR = total revenue per month (simplified for non-SaaS too)
# ============================================================

mrr = (
    orders
    .groupby('order_month')['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'mrr'})
)

# Month-over-month growth — interviewers always ask for this
mrr['mom_growth'] = mrr['mrr'].pct_change()

# Cumulative revenue — shows trajectory
mrr['cumulative_revenue'] = mrr['mrr'].cumsum()

print("MRR Trend:")
mrr

MRR Trend:


,order_month,mrr,mom_growth,cumulative_revenue
0,2024-01,54184.98,NaN,54184.98
1,2024-02,135088.13,1.49,189273.11
2,2024-03,212060.58,0.57,401333.69
3,2024-04,229072.94,0.08,630406.63
4,2024-05,240420.31,0.05,870826.94
5,2024-06,216630.31,-0.10,1087457.25
6,2024-07,163921.37,-0.24,1251378.62
7,2024-08,106368.92,-0.35,1357747.54
8,2024-09,14644.77,-0.86,1372392.31
